# SIB - Exercício 7.3

Teste da classe `KNNRegressor` com o dataset cpu.csv (problema de regressão).

In [1]:
import numpy as np

from si.io.csv_file import read_csv
from si.model_selection.split import train_test_split
from si.models.knn_regressor import KNNRegressor
from si.metrics.rmse import rmse

In [2]:
cpu = read_csv('../datasets/cpu/cpu.csv', sep=',', features=True, label=True)

print('dimensões:', cpu.shape())
print('features:', cpu.features)
print('label:', cpu.label)
print('y -> média: {:.2f} | mínimo: {:.0f} | máximo: {:.0f}'.format(cpu.y.mean(), cpu.y.min(), cpu.y.max()))

dimensões: (209, 6)
features: ['syct', 'mmin', 'mmax', 'cach', 'chmin', 'chmax']
label: perf
y -> média: 105.62 | mínimo: 6 | máximo: 1150


O `y` (`perf`) é contínuo, por isso este é um problema de regressão e a métrica é o RMSE, não a accuracy. Pela mesma razão não se usa aqui a divisão estratificada: não há classes para estratificar.

In [3]:
cpu.summary()

,syct,mmin,mmax,cach,chmin,chmax
mean,203.822967,2.867981e+03,1.179615e+04,25.205742,4.698565,18.267943
median,110.000000,2.000000e+03,8.000000e+03,8.000000,2.000000,8.000000
min,17.000000,6.400000e+01,6.400000e+01,0.000000,0.000000,0.000000
max,1500.000000,3.200000e+04,6.400000e+04,256.000000,52.000000,176.000000
var,67412.691147,1.497266e+07,1.368544e+08,1642.794991,46.239280,672.626771


## Treino e avaliação

In [4]:
train, test = train_test_split(cpu, test_size=0.2, random_state=42)
print('treino:', train.shape(), '| teste:', test.shape())

model = KNNRegressor(k=3)
model.fit(train)

predictions = model.predict(test)
print('\nprimeiras 5 predições:', np.round(predictions[:5], 2))
print('valores reais      :', test.y[:5])
print('\nRMSE:', round(model.score(test), 3))

treino: (168, 6) | teste: (41, 6)

primeiras 5 predições: [140.67  29.33  35.67 701.33  18.67]
valores reais      : [274  30  22 915  16]

RMSE: 81.363


## Efeito do k

In [5]:
for k in (1, 2, 3, 5, 10, 20):
    model = KNNRegressor(k=k)
    model.fit(train)
    print(f'k = {k:2d} -> RMSE = {model.score(test):.3f}')

k =  1 -> RMSE = 100.574
k =  2 -> RMSE = 61.367
k =  3 -> RMSE = 81.363
k =  5 -> RMSE = 104.974
k = 10 -> RMSE = 132.536
k = 20 -> RMSE = 156.322


## Leitura dos resultados

O melhor resultado é k = 2, com um RMSE de cerca de 61, num `y` cuja média é 105.6. Mesmo no melhor caso o erro típico é mais de metade da grandeza que se quer prever, ou seja, o desempenho é fraco.

A causa principal é a escala das features. A coluna `mmax` chega aos 32000 enquanto a `chmin` não passa das dezenas: na distância euclidiana, a soma dos quadrados é dominada pela `mmax` e as restantes features praticamente não influenciam a escolha dos vizinhos. Normalizar as features antes do treino (por exemplo com um StandardScaler) resolveria este problema.

O RMSE não desce de forma monótona com o k. Com k = 1 o modelo fica preso ao vizinho mais próximo e sensível ao ruído; a partir de k = 5 as predições aproximam-se da média global e perdem capacidade de distinguir amostras, com o erro a subir de forma consistente até k = 20.